In [0]:
-- =====================================================================
-- Recrear Silver (catálogo y reviews de Steam) y Gold dim_game desde Bronze
-- ---------------------------------------------------------------------
-- Ejecutar en un SQL editor o en una celda %sql de Databricks, en orden.
-- Lee TODO lo que hay en Bronze (sin fecha de corte) y reescribe las
-- tablas con CREATE OR REPLACE / INSERT OVERWRITE. Es seguro repetirlo.
-- Misma lógica que src/transformation/silver_games.ipynb,
-- src/transformation/silver_game_reviews.ipynb y src/gold_tables.ipynb.
-- =====================================================================

USE CATALOG steam_game_intelligence;

-- ---------------------------------------------------------------------
-- 0) Comprobación previa: Bronze debe tener filas
-- ---------------------------------------------------------------------
SELECT 'bronze.steam_games_batch' AS tabla, count(*) AS filas FROM bronze.steam_games_batch
UNION ALL
SELECT 'bronze.steam_game_reviews_batch', count(*) FROM bronze.steam_game_reviews_batch;


-- ---------------------------------------------------------------------
-- 1) silver.steam_games: una fila válida por app_id (la más reciente)
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE silver.steam_games AS
WITH typed AS (
  SELECT
    try_cast(app_id AS BIGINT) AS app_id,
    name,
    lower(trim(name)) AS name_normalized,
    try_to_date(release_date) AS release_date,
    try_cast(price AS DECIMAL(18,2)) AS price,
    price_status,
    estimated_owners,
    try_cast(regexp_extract(estimated_owners, '^([0-9]+)', 1) AS DECIMAL(38,0)) AS owners_min,
    try_cast(regexp_extract(estimated_owners, '([0-9]+)[^0-9]*$', 1) AS DECIMAL(38,0)) AS owners_max,
    developers, publishers,
    from_json(genres, 'array<string>') AS genres,
    from_json(categories, 'array<string>') AS categories,
    from_json(tags, 'array<string>') AS tags,
    try_cast(positive AS BIGINT) AS positive,
    try_cast(negative AS BIGINT) AS negative,
    try_cast(recommendations AS DECIMAL(38,0)) AS recommendations,
    try_cast(peak_ccu AS BIGINT) AS peak_ccu,
    metacritic_score, user_score,
    average_playtime_forever, median_playtime_forever,
    average_playtime_2weeks, median_playtime_2weeks,
    short_description, about_the_game, detailed_description, notes,
    try_cast(achievements AS DECIMAL(38,0)) AS achievements,
    dlc_count, packages, supported_languages, full_audio_languages,
    windows, mac, linux, header_image, screenshots, movies, website,
    support_url, support_email, metacritic_url,
    steam_store_available, steam_spy_available, _ingested_at, _source
  FROM bronze.steam_games_batch
), ranked AS (
  SELECT *, row_number() OVER (
    PARTITION BY app_id
    ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC
  ) AS rn
  FROM typed
), candidate AS (
  SELECT *, filter(array(
    CASE WHEN app_id IS NULL OR app_id <= 0 THEN 'app_id_invalido' END,
    CASE WHEN name IS NULL OR trim(name) = '' THEN 'name_vacio' END,
    CASE WHEN price < 0 THEN 'price_negativo' END,
    CASE WHEN positive < 0 OR negative < 0 OR peak_ccu < 0 THEN 'conteo_negativo' END,
    CASE WHEN estimated_owners IS NOT NULL AND trim(estimated_owners) <> ''
              AND (owners_min IS NULL OR owners_max IS NULL) THEN 'estimated_owners_invalido' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked
  WHERE rn = 1
)
SELECT * EXCEPT (rn, error_reasons)
FROM candidate
WHERE size(error_reasons) = 0;


-- ---------------------------------------------------------------------
-- 2) silver.steam_game_reviews: una fila válida por app_id (la más reciente)
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE silver.steam_game_reviews AS
WITH typed AS (
  SELECT
    try_cast(app_id AS BIGINT) AS app_id,
    trim(name) AS name,
    trim(reviews) AS reviews,
    _ingested_at,
    trim(_source) AS _source
  FROM bronze.steam_game_reviews_batch
), ranked AS (
  SELECT *, row_number() OVER (
    PARTITION BY app_id
    ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC
  ) AS rn
  FROM typed
)
SELECT app_id, name, reviews, _ingested_at, _source
FROM ranked
WHERE rn = 1
  AND app_id IS NOT NULL AND app_id > 0
  AND name IS NOT NULL AND name <> ''
  AND reviews IS NOT NULL AND reviews <> ''
  AND _ingested_at IS NOT NULL
  AND _source IS NOT NULL AND _source <> '';


-- ---------------------------------------------------------------------
-- 3) gold.dim_game: dimensión de juegos desde silver.steam_games
-- ---------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS gold.dim_game (
  app_id BIGINT, name STRING, release_date DATE, is_unreleased BOOLEAN,
  price DECIMAL(18,2), owners_min DECIMAL(38,0), owners_max DECIMAL(38,0),
  positive BIGINT, negative BIGINT, review_ratio DOUBLE,
  genres ARRAY<STRING>, developers STRING, publishers STRING
) USING DELTA;

INSERT OVERWRITE TABLE gold.dim_game
SELECT
  app_id,
  name,
  release_date,
  CASE WHEN release_date > current_date() THEN true ELSE false END AS is_unreleased,
  price,
  owners_min,
  owners_max,
  positive,
  negative,
  try_divide(positive, positive + negative) AS review_ratio,
  genres,
  developers,
  publishers
FROM silver.steam_games;


-- ---------------------------------------------------------------------
-- 4) Verificación: se esperan ~144 mil juegos en silver.steam_games y dim_game
-- ---------------------------------------------------------------------
SELECT 'silver.steam_games' AS tabla, count(*) AS filas FROM silver.steam_games
UNION ALL
SELECT 'silver.steam_game_reviews', count(*) FROM silver.steam_game_reviews
UNION ALL
SELECT 'gold.dim_game', count(*) FROM gold.dim_game;

-- ---------------------------------------------------------------------
-- NOTA: gold.game_growth (release_date, months_since_release) y
-- gold.twitch_steam_relationship (cruce por nombre) también leen
-- silver.steam_games. Para que los modelos de ML usen el catálogo
-- completo, vuelve a ejecutar src/gold_tables.ipynb después de este script
-- (o el orquestador completo).
-- ---------------------------------------------------------------------
